# ⚙️ Configuration & Export

How to configure parsing: mask proper nouns, remove broken words, and build multi-project datasets.

## 🎯 Goals of this notebook

1. **Understand RunConfig options** — learn every toggle the parser exposes
2. **Learn how to drop broken or damaged texts** — see the difference between word-level and sign-level filtering
3. **See the effects of masking** — see the proper nouns available to mask, understand the limitations
4. **Build a multi-project dataset** — parse several corpora and combine into one DataFrame
5. **Quick analysis** — explore the combined dataset by project, provenance, and period

## Prerequisite: install the package

In [1]:
# Install oracc-parser from PyPI
!pip install oracc-parser -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.3/140.3 kB 1.0 MB/s eta 0:00:00


## Prerequisite: set up the data directory

A DATA_DIR must be defined for the package to work correctly. This is where the data is stored after download from Zenodo or the live ORACC server.

**Option 1**: Colab

Set it up as a folder in your Google Drive (requires mounting Google Drive).

**Option 2**: Run Locally

Set it up anywhere you want on your computer.

Choose whichever option below you prefer (comment out the other one).

In [2]:
# --- Option 1: persist downloaded data across Colab sessions using Google Drive ---
# Without this, data downloads to /content/oracc_data and is lost when the session ends.
# Uncomment the lines below to mount your Drive and store data there persistently.
#
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["ORACC_DATA_DIR"] = "/content/drive/MyDrive/oracc_data"

# --- Option 2: run locally (not in Colab) ---
# If you are running this notebook locally, set the data directory here.
# Uncomment the lines below and set the path to where you want data stored.
#
# import oracc_parser.settings as settings
# from pathlib import Path
# settings.DATA_DIR = Path("/path/to/your/oracc_data")

Mounted at /content/drive


## 1. RunConfig options

These are RunConfig's parameters:

| Option | Default | What it does |
|---|---|---|
| `limit` | `None` | Only parse first N texts (good for testing) |
| `max_break_fraction` | `1.0` | **Word-level**: fraction of broken signs tolerated before a word is replaced with `X` in transliteration / normalization / lemmatization |
| `drop_missing` | `False` | **Sign-level**: replace signs marked `[x]` (completely broken) with `X` in **Unicode cuneiform output only** |
| `drop_damaged` | `False` | **Sign-level**: replace signs marked `⸢x⸣` (partially damaged) with `X` in **Unicode cuneiform output only** |
| `mask_pos` | `[]` | Replace words of certain POS with the tag name |

`limit` was used in Notebook 01. It is intended for testing and demonstration purposes for quicker runs.

The effects of `max_break_fraction`, `drop_missing`, and `drop_damaged` are explained in Section 2. The effects of `mask_pos` is explained in section 3.

## 2. Word-level vs sign-level break filtering

`RunConfig` exposes **two independent levels** of break filtering that operate on different granularities and affect different outputs.

### Word-level — `max_break_fraction` (0.0 – 1.0)

- Each word has a `break_perc`: the **fraction of its signs** that are broken or missing (averaged across all signs in the word).

$\text{break_perc} = \frac{n^{\text{broken signs}} \qquad + \quad 0.5 \cdot {n^{\text{damaged signs}}}}{n^{\text{total signs}}}$

- Words whose `break_perc` **exceeds** `max_break_fraction` are replaced wholesale with `X`.
- Affects → **transliteration**, **normalization**, **lemmatization**.
- `1.0` (default) → keep all words regardless of damage.
- `0.0` → replace any word that has even one broken or damaged sign.

### Sign-level — `drop_missing` / `drop_damaged`

- Operates **sign-by-sign**, not word-by-word.
- `drop_missing=True` replaces signs marked `[x]` (completely lost) with `X`.
- `drop_damaged=True` replaces signs marked `⸢x⸣` (partially legible) with `X`.
- Affects → **Unicode cuneiform output only**.

> ⚠️ **The two levels are independent and produce results that may not align.**
>
> A word kept intact in the transliteration (because its *average* damage is below `max_break_fraction`) may still have individual signs stripped from the Unicode cuneiform output by `drop_missing` / `drop_damaged`, and vice-versa. Do not expect the text outputs and the Unicode cuneiform to be aligned token-for-token when using these parameters.

> ⚠️ **These parameters do not affect translations** — see Notebook 03.

In [3]:
from oracc_parser import parse_project, RunConfig, get_transliterations, get_unicode_texts

PROJECT = "babcity"  # change to another oracc project if desired

# Requires notebook 01 to have been run first — word CSVs must be on disk
# Change 2 to use more or fewer texts
rec_strict = parse_project(PROJECT, config=RunConfig(max_break_fraction=0.3, limit=2))
rec_liberal = parse_project(PROJECT, config=RunConfig(max_break_fraction=1.0, limit=2))

print("=== Transliteration with max_break_fraction=0.3 (strict) ===")
for _, r in get_transliterations(rec_strict).iterrows():
    print(f"  {r['id']}:")
    print(f"    {r['transliteration'][:120]}")
    print(f"    tokens_without_broken: {r['tokens_without_broken']}/{r['total_tokens']}")

print()
print("=== Transliteration with max_break_fraction=1.0 (liberal, default) ===")
for _, r in get_transliterations(rec_liberal).iterrows():
    print(f"  {r['id']}:")
    print(f"    {r['transliteration'][:120]}")
    print(f"    tokens_without_broken: {r['tokens_without_broken']}/{r['total_tokens']}")

print()
print("=== Unicode cuneiform is unaffected by max_break_fraction ===")
print("(drop_missing / drop_damaged control sign-level filtering here)")
for _, r in get_unicode_texts(rec_strict).iterrows():
    print(f"  {r['id']}: {r['unicode'][:80]}")

09:32:51 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
Processing babcity: 100%|██████████| 2/2 [00:00<00:00,  2.57it/s]
09:32:53 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 2 tablets for babcity from word CSVs
09:32:58 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
Processing babcity: 100%|██████████| 2/2 [00:00<00:00, 31.17it/s]
09:32:58 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 2 tablets for babcity from word CSVs


=== Transliteration with max_break_fraction=0.3 (strict) ===
  babcity_P261352:
    X X X X X X X X X
{m}da-ri-ia-⸢a⸣-muš LUGAL ša₂ X
A ša₂ {m}{d}EN.LIL₂-TIN{iṭ} ša₂ ina IGI {m}ti-ri-ra-ka-am-ma
DUMU E₂ š
    tokens_without_broken: 63/88
  babcity_P285916:
    E₂ ša₂ {m}ni-din-ti-{d}EN DUMU {m}ši-rik
ša₂ DA E₂ {m}SILA-a-a
{m}ni-din-tu₄ DUMU {m}ši-rik A {m}e-gi₃-bi
a-na i-di E₂ a
    tokens_without_broken: 91/92

=== Transliteration with max_break_fraction=1.0 (liberal, default) ===
  babcity_P261352:
    1 1/2 MA.NA KU₃.BABBAR i-di E₂ ša₂ MU 1-KAM]
{m}da-ri-ia-⸢a⸣-muš LUGAL ša₂ {[m}{d}EN-it-tan-nu]
A ša₂ {m}{d}EN.LIL₂-TIN{
    tokens_without_broken: 88/88
  babcity_P285916:
    E₂ ša₂ {m}ni-din-ti-{d}EN DUMU {m}ši-rik
ša₂ DA E₂ {m}SILA-a-a
{m}ni-din-tu₄ DUMU {m}ši-rik A {m}e-gi₃-bi
a-na i-di E₂ a
    tokens_without_broken: 92/92

=== Unicode cuneiform is unaffected by max_break_fraction ===
(drop_missing / drop_damaged control sign-level filtering here)
  babcity_P261352: 𒁹 𒈦 𒈠𒈾 𒆬𒌓 𒄿𒁲 

In [4]:
# drop_missing / drop_damaged operate sign-by-sign on Unicode cuneiform only
# They have no effect on transliteration, normalization, or lemmatization

rec_drop_none    = parse_project(PROJECT, config=RunConfig(limit=2, fetch_translations=False, drop_missing=False, drop_damaged=False))
rec_drop_missing = parse_project(PROJECT, config=RunConfig(limit=2, fetch_translations=False, drop_missing=True,  drop_damaged=False))
rec_drop_both    = parse_project(PROJECT, config=RunConfig(limit=2, fetch_translations=False, drop_missing=True,  drop_damaged=True))

print("=== Unicode cuneiform — no filtering ===")
for _, r in get_unicode_texts(rec_drop_none).iterrows():
    print(f"  {r['id']}: {r['unicode'][:80]}")

print()
print("=== Unicode cuneiform — drop_missing=True ===")
for _, r in get_unicode_texts(rec_drop_missing).iterrows():
    print(f"  {r['id']}: {r['unicode'][:80]}")

print()
print("=== Unicode cuneiform — drop_missing=True, drop_damaged=True ===")
for _, r in get_unicode_texts(rec_drop_both).iterrows():
    print(f"  {r['id']}: {r['unicode'][:80]}")

09:33:53 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
Processing babcity: 100%|██████████| 2/2 [00:00<00:00, 32.06it/s]
09:33:53 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 2 tablets for babcity from word CSVs
09:33:54 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
Processing babcity: 100%|██████████| 2/2 [00:00<00:00, 34.63it/s]
09:33:55 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 2 tablets for babcity from word CSVs
09:33:56 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loade

=== Unicode cuneiform — no filtering ===
  babcity_P261352: 𒁹 𒈦 𒈠𒈾 𒆬𒌓 𒄿𒁲 𒂍 𒃻 𒈬 𒁹𒄰
𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡
𒀀 𒃻 𒁹𒀭𒂗𒆤𒁷𒀉 𒃻 𒀸 𒅆 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒌉 𒂍 𒃻 𒁹𒀭𒂗𒆤𒈬𒈬
𒁹𒀭
  babcity_P285916: 𒂍 𒃻 𒁹𒉌𒁷𒋾𒀭𒂗 𒌉 𒁹𒅆𒋆
𒃻 𒁕 𒂍 𒁹𒋻𒀀𒀀
𒁹𒉌𒁷𒌈 𒌉 𒁹𒅆𒋆 𒀀 𒁹𒂊𒐕𒁉
𒀀𒈾 𒄿𒁲 𒂍 𒀀𒈾 𒈬𒀭𒈾
𒌋𒐉 𒂆 𒆬𒌓 𒌓𒌑 𒀀𒈾 𒁹𒆠𒀭𒌓𒁷

=== Unicode cuneiform — drop_missing=True ===
  babcity_P261352: X X XX XX XX X X X XX
𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 XXXXXX
𒀀 𒃻 𒁹𒀭𒂗𒆤𒁷𒀉 𒃻 𒀸 𒅆 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒌉 𒂍 𒃻 𒁹𒀭𒂗𒆤𒈬𒈬
𒁹𒀭
  babcity_P285916: 𒂍 𒃻 𒁹𒉌𒁷𒋾𒀭𒂗 𒌉 𒁹𒅆𒋆
𒃻 𒁕 𒂍 𒁹𒋻𒀀𒀀
𒁹𒉌𒁷𒌈 𒌉 𒁹𒅆𒋆 𒀀 𒁹𒂊𒐕𒁉
𒀀𒈾 𒄿𒁲 𒂍 𒀀𒈾 𒈬𒀭𒈾
𒌋𒐉 𒂆 𒆬𒌓 𒌓𒌑 𒀀𒈾 𒁹𒆠𒀭𒌓𒁷

=== Unicode cuneiform — drop_missing=True, drop_damaged=True ===
  babcity_P261352: X X XX XX XX X X X XX
𒁹𒁕𒊑𒅀X𒈲 𒈗 𒃻 XXXXXX
𒀀 𒃻 𒁹𒀭𒂗𒆤𒁷𒀉 𒃻 𒀸 𒅆 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒌉 𒂍 𒃻 𒁹𒀭𒂗𒆤𒈬𒈬
𒁹𒀭
  babcity_P285916: 𒂍 𒃻 𒁹𒉌𒁷𒋾𒀭𒂗 𒌉 𒁹𒅆𒋆
𒃻 𒁕 𒂍 𒁹𒋻𒀀𒀀
𒁹𒉌𒁷𒌈 𒌉 𒁹𒅆𒋆 𒀀 𒁹𒂊𒐕𒁉
𒀀𒈾 𒄿𒁲 𒂍 𒀀𒈾 𒈬𒀭𒈾
𒌋𒐉 𒂆 𒆬𒌓 𒌓𒌑 𒀀𒈾 𒁹𒆠𒀭𒌓𒁷


## 3. POS-masking

The `mask_pos` parameter replaces words belonging to specified part-of-speech categories with the tag name itself (e.g., `PN`, `DN`). This is useful for creating anonymized or entity-masked training data — for instance, masking all personal names so a model learns to predict structure without memorizing specific individuals.

> ⚠️ **Not all ORACC texts have POS tags** — if a tag is not documented in the original data, the word will not be masked

> ⚠️ **These parameters do not affect translations** — see Notebook 03

Run the code block below to see the available tags to mask. The `count_2026_06` shows how many words in the preprocessed corpus have the POS tag.

> For full explanation on the bundled reference data, include the POS tags, see Notebook 04.


In [11]:
from oracc_parser import reference_data

pos_tags = reference_data.get_pos_tags()
maskable = pos_tags[["POS-tag", "Update to", "Meaning", "count_2026_06"]].fillna("").head(20) # change the number of remove .head(20)
                                                                                              # if you want to see more POS tags
maskable

,POS-tag,Update to,Meaning,count_2026_06
0,,,No tag,1469288
1,N,N,Noun,723178
2,u,U,Unknown,453623
3,V,V,Verb,193607
4,PRP,PRP,Preposition,176529
5,n,NUM,Number,135705
6,PN,PN,Personal Name,105465
7,AJ,AJ,Adjective,74225
8,DET,DET,Determinative Pronoun,63660
9,DN,DN,Divine Name,40016


Run the block below to see the effects of masking.

> ⚠️ **Valid values for `mask_pos` are from the `Update to` column!**

In [12]:
from oracc_parser import parse_project, RunConfig, get_transliterations

PROJECT = "babcity"

# Parse without masking
records_plain  = parse_project(PROJECT, config=RunConfig(limit=3))
# Mask personal and divine names
records_masked = parse_project(PROJECT, config=RunConfig(limit=3, mask_pos=["PN", "DN", "NUM"]))

plain_df  = get_transliterations(records_plain)
masked_df = get_transliterations(records_masked)

for (_, row_p), (_, row_m) in zip(plain_df.iterrows(), masked_df.iterrows()):
    print(f"=== {row_p['id']} ===")
    print(f"  plain:  {row_p['transliteration'][:120]}")
    print(f"  masked: {row_m['transliteration'][:120]}")
    print()

09:46:44 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
Processing babcity: 100%|██████████| 3/3 [00:00<00:00, 33.98it/s]
09:46:44 | INFO    | oracc_parser | Processed 3 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 3 tablets for babcity from word CSVs
09:46:45 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
Processing babcity: 100%|██████████| 3/3 [00:00<00:00, 18.48it/s]
09:46:46 | INFO    | oracc_parser | Processed 3 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 3 tablets for babcity from word CSVs


=== babcity_P261352 ===
  plain:  1 1/2 MA.NA KU₃.BABBAR i-di E₂ ša₂ MU 1-KAM]
{m}da-ri-ia-⸢a⸣-muš LUGAL ša₂ {[m}{d}EN-it-tan-nu]
A ša₂ {m}{d}EN.LIL₂-TIN{
  masked: NUM NUM MA.NA KU₃.BABBAR i-di E₂ ša₂ MU NUM
{m}da-ri-ia-⸢a⸣-muš LUGAL ša₂ PN
A ša₂ PN ša₂ ina IGI PN
DUMU E₂ ša₂ PN
PN {

=== babcity_P285916 ===
  plain:  E₂ ša₂ {m}ni-din-ti-{d}EN DUMU {m}ši-rik
ša₂ DA E₂ {m}SILA-a-a
{m}ni-din-tu₄ DUMU {m}ši-rik A {m}e-gi₃-bi
a-na i-di E₂ a
  masked: E₂ ša₂ PN DUMU PN
ša₂ DA E₂ PN
PN DUMU PN A {m}e-gi₃-bi
a-na i-di E₂ a-na MU.AN.NA
NUM GIN₂ KU₃.BABBAR BABBAR{u₂} a-na P

=== babcity_P295135 ===
  plain:  E₂-⸢su⸣ ša₂ {e₂}a-su-up ša₂ {m}⸢x⸣-[...]
A-šu₂ ša₂ {m}lu-ṣu-a-na-ZALAG₂-{d}AMAR.UTU [...]
a-na MU.AN.NA 2 GIN₂ bit-[qa K
  masked: E₂-⸢su⸣ ša₂ {e₂}a-su-up ša₂ PN
A-šu₂ ša₂ PN [...]
a-na MU.AN.NA NUM GIN₂ bit-[qa KU₃.BABBAR a-na]
i-di E₂ a-na PN
A-šu₂ 



## 4. Build a multi-project dataset

Using the functions introduced thus far, you can combine the output of `get_full_flat_table(records)` to prepare a machine-learning ready dataset that spans across multiple ORACC projects.

Run the code below to see the results, you can change the project and the `RunConfig` limit on texts processed.

In [14]:
from oracc_parser.pipeline import get_full_flat_table
from oracc_parser import parse_project, RunConfig
import pandas as pd

# Change these to any projects you want
PROJECTS = ["babcity", "borsippa"]
config = RunConfig(max_break_fraction=0.5, drop_missing=True, mask_pos=["PN", "DN", "NUM"], limit=3)  # change 3 to use more or fewer texts

all_dfs = []
for project in PROJECTS:
    print(f"Parsing {project}...")
    try:
        records = parse_project(project, config=config)
        df = get_full_flat_table(records)
        all_dfs.append(df)
        print(f"  → {len(records)} tablets")
    except Exception as e:
        print(f"  → Error: {e}")

combined = pd.concat(all_dfs, ignore_index=True)
print(f"\n✓ Combined: {len(combined)} tablets from {len(PROJECTS)} projects")
combined

Parsing babcity...


09:50:48 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
Processing babcity: 100%|██████████| 3/3 [00:00<00:00, 27.92it/s]
09:50:48 | INFO    | oracc_parser | Processed 3 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 3 tablets for babcity from word CSVs


  → 3 tablets
Parsing borsippa...


09:50:50 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/borsippa
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/borsippa
Processing borsippa: 100%|██████████| 3/3 [00:00<00:00, 15.26it/s]
09:50:50 | INFO    | oracc_parser | Processed 3 tablets for borsippa from word CSVs
INFO:oracc_parser:Processed 3 tablets for borsippa from word CSVs


  → 3 tablets

✓ Combined: 6 tablets from 2 projects


,id,project,text_id,genre,archive,provenance,pleiades_id,state_supergroup,period,start_year,...,secondary_literature,credits,cite_as,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,babcity_P261352,babcity,P261352,Legal,Murašû Archive,Nippur,912910,Mix,achaemenid,-547,...,Clay Aram.Endors. 306 #17(Aram.only); PBS 02/1...,Based on a transliteration by Heather D. Baker...,Please cite this page as http://oracc.org/babc...,X X X X X X X X X\n{m}da-ri-ia-⸢a⸣-muš LUGAL š...,X X X X X X X X X\nDarius šarri ša X\nmāru ša ...,X X X X X X X X X\nDarius šarru ša X\nmāru ša ...,X X XX XX XX X X X XX\n𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 XXXXXX\n𒀀 𒃻 ...,,88,64
1,babcity_P285916,babcity,P285916,Legal,Egibi Archive,Babylon,893951,Mix,achaemenid,-547,...,"Zawadzki, Rental of Houses 130","Based on the edition of Stefan Zawadzki, The R...",Please cite this page as http://oracc.org/babc...,E₂ ša₂ PN DUMU PN\nša₂ DA E₂ PN\nPN DUMU PN A ...,bītu ša PN māri PN\nša ṭāh bīt PN\nPN mār PN m...,bītu ša PN māru PN\nša ṭāh bītu PN\nPN māru PN...,𒂍 𒃻 𒁹𒉌𒁷𒋾𒀭𒂗 𒌉 𒁹𒅆𒋆\n𒃻 𒁕 𒂍 𒁹𒋻𒀀𒀀\n𒁹𒉌𒁷𒌈 𒌉 𒁹𒅆𒋆 𒀀 𒁹𒂊𒐕...,,92,92
2,babcity_P295135,babcity,P295135,Legal,Ea-ilutu-bani Archive,Borsippa,893964,Mix,achaemenid,-547,...,"Joannes,Archives de Borsippa 94 (Tr) and 325 (T)","Adapted, lemmatised and translated by Lindsey ...",Please cite this page as http://oracc.org/babc...,E₂-⸢su⸣ ša₂ {e₂}a-su-up ša₂ PN\nA-šu₂ ša₂ PN X...,bīssu ša asup ša PN\nmārišu ša PN X\nana šatti...,bītu ša asuppu ša PN\nmāru ša PN X\nana šattu ...,𒂍𒋢 𒃻 𒂍𒀀𒋢𒌒 𒃻 𒁹XX\n𒀀𒋙 𒃻 𒁹𒇻𒍮𒀀𒈾𒂟𒀭𒀫𒌓 X\n𒀀𒈾 𒈬𒀭𒈾 𒈫 𒂆 ...,,84,70
3,borsippa_P202111,borsippa,P202111,Legal,Ilia Archive,Borsippa,893964,Mix,achaemenid,-547,...,,"Adapted from Caroline Waerzeggers, The Ezida T...",Please cite this page as http://oracc.org/bors...,NUM NUM u₄-mu maš-ti-ti ša₂ {iti}GUD NUM NUM u...,NUM NUM ūmu maštīti ša Ayyaru NUM NUM ūmu\nmaš...,NUM NUM ūmu maštītu ša Ayyaru NUM NUM ūmu\nmaš...,𒈫 𒈦 𒌓𒈬 𒈦𒋾𒋾 𒃻 𒌗𒄞 𒈫 𒈦 𒌓𒈬\nX𒋾𒋾 𒌗𒆥 𒈫 𒈦 𒌓𒈬 𒆏𒊑 𒃻 𒌗𒃶\...,,102,91
4,borsippa_P202351,borsippa,P202351,Legal,Ilia Archive,Borsippa,893964,Mix,Neo-Babylonian,-626,...,,"Adapted from Caroline Waerzeggers, The Ezida T...",Please cite this page as http://oracc.org/bors...,X X X PN A {m}DINGIR-⸢ia⸣\nX X lib₃]-bi-šu₂ PN...,X X X PN mār Ilia\nX X libbišu PN qallašu\nX m...,X X X PN māru Ilia\nX X libbu PN qallu\nX māru...,XXXX XX X 𒁹𒉣𒆷𒀀 𒀀 𒁹𒀭𒅀\nX XX X𒁉𒋙 𒁹𒆗𒁀𒀀 𒇽𒃲𒆷X\nXX 𒇽...,,157,96
5,borsippa_P202504,borsippa,P202504,Legal,Ahiya’ūtu Archive,Borsippa,893964,Mix,achaemenid,-547,...,,"Adapted from Caroline Waerzeggers, The Ezida T...",Please cite this page as http://oracc.org/bors...,X u₄]-mu tar-din-nu {iti}ŠE TA UD NUM\nX ⸢UD⸣ ...,X ūmu tardinnu Addaru ultu ūm NUM\nX ūm NUM NU...,X ūmu tardennu Addaru ištu ūmu NUM\nX ūmu NUM ...,X X𒈬 𒋻𒁷𒉡 𒌗𒊺 𒋫 𒌓 X𒄰\nXX 𒌓 𒌋𒐋𒄰 𒐈 𒌓𒈬 𒋾𒅆𒆕𒈨𒌍\nX𒁈 𒌓 ...,,247,181


In [15]:
from oracc_parser.settings import OUTPUT_DIR

# Export the combined dataset
out = OUTPUT_DIR
out.mkdir(parents=True, exist_ok=True)

# change the file names here, if you want
path_jsonl = out / "combined_dataset.jsonl"
path_csv = out / "combined_dataset.csv"

combined.to_json(path_jsonl, orient="records", lines=True, force_ascii=False)
combined.to_csv(path_csv, index=False)

print(f"Saved to:")
print(f"  {path_jsonl}  ({path_jsonl.stat().st_size/1024:.1f} KB)")
print(f"  {path_csv}  ({path_csv.stat().st_size/1024:.1f} KB)")
print(f"\n📁 All output files in {out}:")
for f in sorted(out.iterdir()):
    if f.is_file():
        print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

Saved to:
  /content/drive/MyDrive/oracc_data/output/combined_dataset.jsonl  (25.4 KB)
  /content/drive/MyDrive/oracc_data/output/combined_dataset.csv  (23.1 KB)

📁 All output files in /content/drive/MyDrive/oracc_data/output:
  babcity.csv                               41.7 KB
  babcity.jsonl                             45.8 KB
  combined_dataset.csv                      23.1 KB
  combined_dataset.jsonl                    25.4 KB


## 5. Quick analysis of the data

In [16]:
print("Texts by project:")
print(combined["project"].value_counts().to_string())

print("\nTexts by provenance:")
print(combined["provenance"].value_counts().to_string())

print("\nTexts by period:")
print(combined["period"].value_counts().to_string())

print(f"\nAvg tokens per text: {combined['total_tokens'].mean():.0f}")
print(f"Total tokens: {combined['total_tokens'].sum()}")

Texts by project:
project
babcity     3
borsippa    3

Texts by provenance:
provenance
Borsippa    4
Nippur      1
Babylon     1

Texts by period:
period
achaemenid        5
Neo-Babylonian    1

Avg tokens per text: 128
Total tokens: 770


## What's next?

- **Explore what's in the dataset:** See notebook `02_reference_data.ipynb` for a full overview of collected projects, catalogues, provenance, periods, and other reference data.
- **Configure parsing:** See notebook `03_configure_and_export.ipynb` for masking PNs, dropping broken signs, and changing output formats.
- **understand the process:** see notebook `04_oracc_json_processing.ipynb` for in-depth explanations on how the package processes the original oracc files and metadata.